# Combining Raw Data for AI Stock Prediction

Based on Chapter 4 of *Build Your Own AI Investor* by Damon Lee.

**Pipeline overview**

| Step | Description |
|------|-------------|
| 1 | Load financial fundamentals (Income Statement, Balance Sheet, Cash Flow) from SimFin |
| 2 | Load daily share prices |
| 3 | Match stock prices to report dates and compute annual performance |
| 4 | Filter rows with missing / low-quality data |
| 5 | Save cleaned datasets for model training |

**Part 2** prepares 2024 data for live stock selection, fetching latest prices via *yfinance*.

In [ ]:
# =============================================================================
# Configuration  -  edit paths and parameters here
# =============================================================================
from pathlib import Path

# SimFin bulk-download directory
DATA_DIR = Path(
    "C:/Users/damon/OneDrive/BYO_Investing_AI/2024/Stock_Data/SimFin2024/"
)

# Output CSVs
FUNDAMENTALS_RAW_CSV        = "Annual_Stock_Price_Fundamentals.csv"
PERFORMANCE_RAW_CSV         = "Annual_Stock_Price_Performance.csv"
FUNDAMENTALS_FILTERED_CSV   = "Annual_Stock_Price_Fundamentals_Filtered.csv"
PERFORMANCE_FILTERED_CSV    = "Annual_Stock_Price_Performance_Filtered.csv"
FUNDAMENTALS_2024_CSV       = "Annual_Stock_Price_Fundamentals_Filtered_2024_present.csv"
TICKERS_2024_CSV            = "Tickers_Dates_2024_present.csv"

# Parameters
PERFORMANCE_HORIZON_DAYS = 365   # forward-looking window for Y target
PRICE_WINDOW_DAYS        = 5     # trading-day search window around a date
MIN_DOLLAR_VOLUME        = 1e4   # minimum dollar-volume filter

# 2024 stock-selection date range
SELECTION_DATE_START = "2024-01-01"
SELECTION_DATE_END   = "2024-04-01"

In [ ]:
import math
from typing import Optional

import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

# Plotting defaults
plt.rcParams["figure.figsize"] = [7.0, 4.5]
plt.rcParams["figure.dpi"] = 150

## Data Loading Functions

In [ ]:
def load_fundamentals(data_dir: Path = DATA_DIR) -> pd.DataFrame:
    """Load and merge Income Statement, Balance Sheet, and Cash Flow CSVs from SimFin.

    Returns a single DataFrame keyed on
    (Ticker, SimFinId, Currency, Fiscal Year, Report Date, Publish Date).
    """
    merge_keys = [
        "Ticker", "SimFinId", "Currency",
        "Fiscal Year", "Report Date", "Publish Date",
    ]

    income   = pd.read_csv(data_dir / "us-income-annual-full-asreported.csv",   delimiter=";")
    balance  = pd.read_csv(data_dir / "us-balance-annual-full-asreported.csv",  delimiter=";")
    cashflow = pd.read_csv(data_dir / "us-cashflow-annual-full-asreported.csv", delimiter=";")

    print(f"Income Statement : {income.shape}")
    print(f"Balance Sheet    : {balance.shape}")
    print(f"Cash Flow        : {cashflow.shape}")

    result = (
        income
        .merge(balance,  on=merge_keys)
        .merge(cashflow, on=merge_keys)
    )

    result["Report Date"]  = pd.to_datetime(result["Report Date"])
    result["Publish Date"] = pd.to_datetime(result["Publish Date"])

    print(f"Merged result    : {result.shape}")
    return result

In [ ]:
def load_share_prices(data_dir: Path = DATA_DIR) -> pd.DataFrame:
    """Load daily US share prices from a SimFin CSV.

    Returns a DataFrame with a parsed ``Date`` column.
    """
    prices = pd.read_csv(data_dir / "us-shareprices-daily.csv", delimiter=";")
    prices["Date"] = pd.to_datetime(prices["Date"])
    print(f"Share prices     : {prices.shape}")
    return prices

## Price Matching Functions

In [ ]:
def get_price_near_date(
    ticker: str,
    date,
    offset_days: int,
    prices: pd.DataFrame,
    window_days: int = PRICE_WINDOW_DAYS,
) -> dict:
    """Return the first available Open price and dollar-volume within a date window.

    Searches for trades between ``date + offset_days`` and
    ``date + offset_days + window_days``.

    Returns a dict with keys: Ticker, Open, Date, DollarVolume.
    Values are NaN / NaT when no trade is found.
    """
    start = pd.to_datetime(date) + pd.Timedelta(days=offset_days)
    end   = start + pd.Timedelta(days=window_days)

    rows = prices[
        (prices["Ticker"] == ticker) & prices["Date"].between(start, end)
    ]

    if rows.empty:
        return {
            "Ticker": ticker,
            "Open": np.nan,
            "Date": pd.NaT,
            "DollarVolume": np.nan,
        }

    first = rows.iloc[0]
    return {
        "Ticker": ticker,
        "Open": first["Open"],
        "Date": first["Date"],
        "DollarVolume": first["Volume"] * first["Open"],
    }

In [ ]:
def get_annual_performance(
    fundamentals: pd.DataFrame,
    prices: pd.DataFrame,
    horizon_days: int = PERFORMANCE_HORIZON_DAYS,
    date_col: str = "Publish Date",
) -> pd.DataFrame:
    """Look up the share price at each report's publish date *and* at
    publish date + ``horizon_days``.

    Warning
    -------
    Row-by-row lookup — takes **several hours** on large datasets.
    """
    records = []
    n = len(fundamentals)

    for i, (_, row) in enumerate(fundamentals.iterrows()):
        ticker = row["Ticker"]
        date   = row[date_col]

        start_info = get_price_near_date(ticker, date, 0, prices)
        end_info   = get_price_near_date(ticker, date, horizon_days, prices)

        records.append({
            "Ticker":       start_info["Ticker"],
            "Open Price":   start_info["Open"],
            "Date":         start_info["Date"],
            "Volume":       start_info["DollarVolume"],
            "Ticker2":      end_info["Ticker"],
            "Open Price2":  end_info["Open"],
            "Date2":        end_info["Date"],
            "Volume2":      end_info["DollarVolume"],
        })

        if (i + 1) % 5_000 == 0:
            print(f"  Processed {i + 1:,} / {n:,} rows ...")

    print(f"  Done — {n:,} rows processed.")
    return pd.DataFrame(records)

## Data Filtering & Feature Engineering

In [ ]:
def filter_training_data(
    fundamentals: pd.DataFrame,
    performance: pd.DataFrame,
    min_dollar_volume: float = MIN_DOLLAR_VOLUME,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Remove rows with missing prices, low volume, missing target dates,
    or missing share counts.  Adds a *Market Cap* column.

    Returns
    -------
    (filtered_fundamentals, filtered_performance)
    """
    has_price       = performance["Open Price"].notna()
    has_volume      = (
        (performance["Volume"] >= min_dollar_volume)
        & (performance["Volume2"] >= min_dollar_volume)
    )
    has_target_date = performance["Date2"].notna()

    mask = has_price & has_volume & has_target_date

    fundamentals = fundamentals.loc[mask].copy()
    performance  = performance.loc[mask].copy()

    has_shares = fundamentals["Shares (Diluted)_x"].notna()
    fundamentals = fundamentals.loc[has_shares].copy()
    performance  = performance.loc[has_shares].copy()

    fundamentals = fundamentals.reset_index(drop=True)
    performance  = performance.reset_index(drop=True)

    # Feature: Market Cap = share price x diluted shares
    fundamentals["Market Cap"] = (
        performance["Open Price"] * fundamentals["Shares (Diluted)_x"]
    )

    print(f"After filtering: fundamentals {fundamentals.shape}, "
          f"performance {performance.shape}")
    return fundamentals, performance

---
# Part 1 — Build Historical Training Data

> **Warning:** The price-lookup step iterates row-by-row and takes
> **several hours**.  If you already have saved CSVs, skip ahead to
> *Load from Saved Files*.

In [ ]:
# Step 1 ── Load fundamentals & share prices
fundamentals = load_fundamentals()
share_prices = load_share_prices()

# Step 2 ── Compute annual performance (⚠ slow — several hours)
performance = get_annual_performance(fundamentals, share_prices)

# Step 3 ── Save raw results
fundamentals.to_csv(FUNDAMENTALS_RAW_CSV)
performance.to_csv(PERFORMANCE_RAW_CSV)
print(f"\nSaved: {FUNDAMENTALS_RAW_CSV}, {PERFORMANCE_RAW_CSV}")

### Load from Saved Files & Filter

Run this cell **instead** of the one above if you already have the raw CSVs.

In [ ]:
# Load previously saved raw data
fundamentals = pd.read_csv(FUNDAMENTALS_RAW_CSV, index_col=0)
performance  = pd.read_csv(PERFORMANCE_RAW_CSV,  index_col=0)

# Filter & engineer features
fundamentals, performance = filter_training_data(fundamentals, performance)

# Quick visualisation of publication-date distribution
fundamentals["Publish Date"] = pd.to_datetime(fundamentals["Publish Date"])
fundamentals[
    fundamentals["Publish Date"].between("2004-01-01", "2023-06-01")
]["Publish Date"].hist(bins=200, figsize=(10, 5))
plt.title("USA Financial Report Publication Dates")
plt.xlabel("Date")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

# Save filtered data
fundamentals.to_csv(FUNDAMENTALS_FILTERED_CSV)
performance.to_csv(PERFORMANCE_FILTERED_CSV)
print(f"Saved: {FUNDAMENTALS_FILTERED_CSV}, {PERFORMANCE_FILTERED_CSV}")

---
# Part 2 — 2024 Stock Selection Data

Fetch the latest fundamentals (published Jan–Apr 2024), look up current
share prices via **yfinance**, and prepare a dataset for stock selection.

In [ ]:
# Load fundamentals & filter to the 2024 publication window
fundamentals_2024 = load_fundamentals()

fundamentals_2024 = fundamentals_2024[
    fundamentals_2024["Publish Date"].between(
        SELECTION_DATE_START, SELECTION_DATE_END
    )
].reset_index(drop=True)

print(f"2024 fundamentals after date filter: {fundamentals_2024.shape}")

In [ ]:
import yfinance as yf


def fetch_latest_prices(tickers: pd.Series) -> dict[str, float]:
    """Fetch the most recent open price for each unique ticker via yfinance."""
    prices_map: dict[str, float] = {}
    for sym in tickers.unique():
        try:
            hist = yf.Ticker(sym).history(period="1d")
            if not hist.empty:
                prices_map[sym] = hist["Open"].iloc[0]
                print(f"  {sym}: {prices_map[sym]:.2f}")
        except Exception as exc:
            print(f"  {sym}: FAILED ({exc})")
    return prices_map


latest_prices = fetch_latest_prices(fundamentals_2024["Ticker"])
print(f"\nObtained prices for {len(latest_prices)} tickers.")

In [ ]:
# Merge latest prices onto fundamentals by ticker
latest_prices_df = pd.DataFrame(
    list(latest_prices.items()), columns=["Ticker", "Latest Open Price"]
)
fundamentals_2024 = fundamentals_2024.merge(latest_prices_df, on="Ticker", how="left")

# Filter: require valid price and share count
has_price  = fundamentals_2024["Latest Open Price"].notna()
has_shares = fundamentals_2024["Shares (Diluted)_x"].notna()
fundamentals_2024 = fundamentals_2024[has_price & has_shares].reset_index(drop=True)

# Market Cap using latest share price
fundamentals_2024["Market Cap"] = (
    fundamentals_2024["Latest Open Price"]
    * fundamentals_2024["Shares (Diluted)_x"]
)

# Build a tickers reference table
tickers_2024 = fundamentals_2024[["Ticker", "Latest Open Price"]].copy()
tickers_2024.rename(columns={"Latest Open Price": "Open Price"}, inplace=True)
tickers_2024["Date"] = pd.Timestamp.today().strftime("%Y-%m-%d")

# Save
fundamentals_2024.drop(columns=["Latest Open Price"]).to_csv(FUNDAMENTALS_2024_CSV)
tickers_2024.to_csv(TICKERS_2024_CSV)
print(f"Saved: {FUNDAMENTALS_2024_CSV}  ({fundamentals_2024.shape})")
print(f"Saved: {TICKERS_2024_CSV}  ({tickers_2024.shape})")